# RQ3 Final Results Packaging

This notebook does not refit RQ3 models. It reads the clean model outputs already generated by the main-analysis and SI notebooks, then creates a compact publication-oriented results package:

- `Results/RQ3_social_determinants/Tables`: only the most central main-text tables.
- `Results/RQ3_social_determinants/Figures`: polished main-text figures.
- `Results/RQ3_social_determinants/Tables/SI` and `Figures/SI`: compact SI tables/figures for diagnostics and robustness.

The visual style follows the RQ1/RQ2 palette, using the existing region colors.

In [ ]:

from pathlib import Path
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from PIL import Image

warnings.filterwarnings('ignore')

PROJECT = Path('/Users/nico/Research/Energy sufficiency')
RQ3_DIR = PROJECT / 'Results/RQ3_social_determinants'
SOURCE_MAIN = RQ3_DIR / 'draft/Main'
SOURCE_SI = RQ3_DIR / 'draft/SI'
SOURCE_SELECTION = RQ3_DIR / 'draft/Main_model_selection'
OUT_TABLE = RQ3_DIR / 'Tables'
OUT_FIG = RQ3_DIR / 'Figures'
OUT_TABLE_SI = OUT_TABLE / 'SI'
OUT_FIG_SI = OUT_FIG / 'SI'
PROCESS_DIR = PROJECT / 'Process/RQ3_final_results_packaging'
for d in [OUT_TABLE, OUT_FIG, OUT_TABLE_SI, OUT_FIG_SI, PROCESS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

palette_path = PROJECT / 'Results/RQ1/Tables/rq1_region_palette.csv'
region_palette = pd.read_csv(palette_path)
region_order = region_palette['analytical_region'].tolist()
region_colors = dict(zip(region_palette['analytical_region'], region_palette['color_hex']))
short_labels = dict(zip(region_palette['analytical_region'], region_palette['short_label']))

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 320,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'legend.title_fontsize': 9,
})
sns.set_theme(style='whitegrid', font='DejaVu Sans')

BLUE = '#53a8e1'
ORANGE = '#ffa74f'
RED = '#ed5f5f'
PURPLE = '#9a72c7'
TEAL = '#2f6f73'
GREY = '#c9cdd2'
DARK = '#2b2f33'
LIGHT_GREY = '#f2f4f6'
print('Output tables:', OUT_TABLE)
print('Output figures:', OUT_FIG)


In [ ]:

# Load clean result tables.
global_m4 = pd.read_csv(SOURCE_MAIN / 'Tables/Table_RQ3_main_global_M4_coefficients.csv')
regional_m4 = pd.read_csv(SOURCE_MAIN / 'Tables/Table_RQ3_main_regional_M4_coefficients.csv')
main_summary = pd.read_csv(SOURCE_MAIN / 'Tables/Table_RQ3_main_global_M4_model_summary.csv')
region_interactions = pd.read_csv(SOURCE_SI / 'Tables/Table_SI_RQ3_05_M4_region_interaction_terms_all_8.csv')

si_vif = pd.read_csv(SOURCE_SI / 'Tables/Table_SI_RQ3_00_main_M4_VIF.csv')
si_corr = pd.read_csv(SOURCE_SI / 'Tables/Table_SI_RQ3_00_main_variable_correlations_r.csv', index_col=0)
si_corr_p = pd.read_csv(SOURCE_SI / 'Tables/Table_SI_RQ3_00_main_variable_correlations_p.csv', index_col=0)
si_stability = pd.read_csv(SOURCE_SI / 'Tables/Table_SI_RQ3_01_global_model_stability_coefficients.csv')
si_spatial = pd.read_csv(SOURCE_SI / 'Tables/Table_SI_RQ3_02_spatial_error_sensitivity_M4.csv')
si_moran = pd.read_csv(SOURCE_SI / 'Tables/Table_SI_RQ3_02_spatial_residual_morans_I.csv')
si_frac = pd.read_csv(SOURCE_SI / 'Tables/Table_SI_RQ3_03_fractional_logit_gap_deficit_M4_AME.csv')
si_global_interactions = pd.read_csv(SOURCE_SI / 'Tables/Table_SI_RQ3_04_M4_interaction_terms_all_8.csv')

var_labels = {
    'SES_raw': 'SES',
    'rural': 'Rurality',
    'education': 'Education',
    'female_share': 'Female share',
    'log_GDP_pc': 'GDP pc',
    'log_population': 'Population',
    'total_HDD': 'HDD',
    'total_CDD': 'CDD',
    'age': 'Age',
    'Hhsize': 'Household size',
}
social_vars = ['SES_raw', 'rural', 'education', 'female_share']
control_vars = ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD']
interaction_order = [
    'Rurality x SES', 'Rurality x Education', 'Female share x SES', 'Female share x Education',
    'SES x Education', 'Age x Female share', 'SES x Household size', 'Female share x Rurality'
]
print(global_m4.shape, regional_m4.shape, region_interactions.shape)


In [ ]:

# Helpers.
def star(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    if p < 0.1: return '†'
    return ''

def fmt_p(p):
    if pd.isna(p): return ''
    if p < 0.001: return '<0.001'
    return f'{p:.3f}'

def fmt_q(q):
    if pd.isna(q): return ''
    if q < 0.001: return 'q<0.001'
    return f'q={q:.3f}'

def fmt_effect(row, effect_col='pct_change_per_sd', low_col='pct_ci_low', high_col='pct_ci_high'):
    s = row.get('stars', '')
    s = '' if pd.isna(s) else str(s)
    return f"{row[effect_col]:+.1f}% [{row[low_col]:+.1f}, {row[high_col]:+.1f}] {s}".strip()

def annotate_effect(ax, x, y, txt, color=DARK, ha='left', va='center', fs=8.5, weight='normal'):
    ax.text(x, y, txt, ha=ha, va=va, fontsize=fs, color=color, fontweight=weight)

def save_table(df, path):
    df.to_csv(path, index=False)
    print(path.name, df.shape)

def clean_axis(ax):
    ax.grid(axis='x', color='#e6e8eb', linewidth=.8)
    ax.grid(axis='y', visible=False)
    ax.tick_params(axis='both', length=0)


In [ ]:

# Main-text tables: keep only the most central three.
main_global_pub = global_m4.copy()
main_global_pub['Variable'] = main_global_pub['variable'].map(var_labels)
main_global_pub['Effect per 1 SD'] = main_global_pub.apply(fmt_effect, axis=1)
main_global_pub['Coefficient'] = main_global_pub['coef_log_points_per_sd'].map(lambda x: f'{x:.3f}')
main_global_pub['SE'] = main_global_pub['cluster_se'].map(lambda x: f'{x:.3f}')
main_global_pub['p'] = main_global_pub['p_value'].map(fmt_p)
main_global_pub = main_global_pub[['Variable', 'Coefficient', 'SE', 'Effect per 1 SD', 'p', 'n_obs', 'n_countries', 'r_squared', 'adj_r_squared']]
save_table(main_global_pub, OUT_TABLE / 'Table_RQ3_1_global_M4_main_results.csv')

regional_social_pub = regional_m4[regional_m4['variable'].isin(social_vars)].copy()
regional_social_pub['Region'] = pd.Categorical(regional_social_pub['region'], categories=region_order, ordered=True)
regional_social_pub['Variable'] = regional_social_pub['variable'].map(var_labels)
regional_social_pub['Effect per 1 SD'] = regional_social_pub.apply(fmt_effect, axis=1)
regional_social_pub['p'] = regional_social_pub['p_value'].map(fmt_p)
regional_social_pub = regional_social_pub.sort_values(['Region', 'variable'])
regional_social_pub = regional_social_pub[['Region', 'Variable', 'Effect per 1 SD', 'p', 'n_obs', 'n_countries']]
save_table(regional_social_pub, OUT_TABLE / 'Table_RQ3_2_regional_M4_social_effects.csv')

exclude_main_interactions = ['Age x Female share', 'SES x Household size']
selected_int = region_interactions[
    (~region_interactions['interaction'].isin(exclude_main_interactions)) &
    (
        (region_interactions['fdr_all_region_interactions_reject_0_05']) |
        ((region_interactions['p_value'] < 0.05) & (region_interactions['interaction'].str.contains('Female share|Education', regex=True)))
    )
].copy()
selected_int['Region'] = pd.Categorical(selected_int['region'], categories=region_order, ordered=True)
selected_int['Interaction'] = selected_int['interaction'].str.replace(' x ', ' × ', regex=False)
selected_int['Coefficient'] = selected_int['coef_interaction_log_points'].map(lambda x: f'{x:+.3f}')
selected_int['95% CI'] = selected_int.apply(lambda r: f"[{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]", axis=1)
selected_int['p'] = selected_int['p_value'].map(fmt_p)
selected_int['FDR q (within region)'] = selected_int['q_value_fdr_within_region_8_tests'].map(fmt_p)
selected_int['FDR q (all region-interactions)'] = selected_int['q_value_fdr_all_region_interactions'].map(fmt_p)
selected_int['FDR status'] = np.where(selected_int['fdr_all_region_interactions_reject_0_05'], 'FDR across all tests', np.where(selected_int['fdr_within_region_reject_0_05'], 'FDR within region only', 'Raw p<0.05 only'))
selected_int['Small-country caution'] = np.where(selected_int['n_countries'] < 10, 'Yes', 'No')
selected_int = selected_int.sort_values(['fdr_all_region_interactions_reject_0_05', 'p_value'], ascending=[False, True])
selected_int = selected_int[['Region', 'Interaction', 'Coefficient', '95% CI', 'p', 'FDR q (within region)', 'FDR q (all region-interactions)', 'FDR status', 'n_obs', 'n_countries', 'Small-country caution']]
save_table(selected_int, OUT_TABLE / 'Table_RQ3_3_region_interactions_selected.csv')


In [ ]:

# Figure 1: polished global M4 coefficients with a stable numeric-results column.
plot = global_m4.copy()
plot['label'] = plot['variable'].map(var_labels)
plot['is_social'] = plot['variable'].isin(social_vars)
plot['order'] = plot['variable'].map({v:i for i,v in enumerate(['SES_raw','rural','education','female_share','log_GDP_pc','log_population','total_HDD','total_CDD'])})
plot = plot.sort_values('order', ascending=False).reset_index(drop=True)
plot['effect_text'] = plot['pct_change_per_sd'].map(lambda x: f'{x:+.1f}%')
plot['stars_clean'] = plot['stars'].apply(lambda x: '' if pd.isna(x) else str(x))

fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
y = np.arange(len(plot))
colors = np.where(plot['is_social'], BLUE, ORANGE)
ax.errorbar(plot['pct_change_per_sd'], y,
            xerr=[plot['pct_change_per_sd'] - plot['pct_ci_low'], plot['pct_ci_high'] - plot['pct_change_per_sd']],
            fmt='none', ecolor='#5f666d', elinewidth=1.25, capsize=3, zorder=1)
ax.scatter(plot['pct_change_per_sd'], y, s=72, color=colors, edgecolor='white', linewidth=.7, zorder=2)
ax.axvline(0, color=DARK, linewidth=.9)
ax.set_yticks(y)
ax.set_yticklabels(plot['label'])
ax.set_xlabel('Percent change in sufficiency ratio per 1 SD increase')
ax.set_title('Global M4 model: social determinants and controls')
clean_axis(ax)

# Keep all numeric labels in one right-side column so CIs and labels cannot collide.
xmin = min(plot['pct_ci_low'].min() - 8, -48)
xmax = max(plot['pct_ci_high'].max() + 18, 74)
ax.set_xlim(xmin, xmax)
x_num = xmax - 12
x_star = xmax - 2.5
ax.text(x_num, len(plot)-.18, 'Effect', ha='right', va='bottom', fontsize=9, fontweight='bold', color=DARK)
for yy, (_, row) in enumerate(plot.iterrows()):
    ax.text(x_num, yy, row['effect_text'], ha='right', va='center', fontsize=9, color=DARK,
            bbox=dict(facecolor='white', edgecolor='none', alpha=.82, pad=1.2))
    if row['stars_clean']:
        ax.text(x_star, yy, row['stars_clean'], ha='left', va='center', fontsize=13, color=RED, fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Social predictor', markerfacecolor=BLUE, markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Control', markerfacecolor=ORANGE, markersize=8),
]
ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=True, framealpha=.95)
fig.savefig(OUT_FIG / 'Figure_RQ3_1_global_M4_coefficients.png', bbox_inches='tight')
plt.show()


In [ ]:

# Figure 2: regional social coefficients. Each panel shows RQ1/RQ2 region labels and matching colors.
regional_plot = regional_m4[regional_m4['variable'].isin(social_vars)].copy()
regional_plot['region'] = pd.Categorical(regional_plot['region'], categories=region_order, ordered=True)
regional_plot['variable_label'] = regional_plot['variable'].map(var_labels)
regional_plot['stars_clean'] = regional_plot['stars'].apply(lambda x: '' if pd.isna(x) else str(x))
regional_plot = regional_plot.sort_values(['region','variable'])
var_order = ['SES_raw', 'rural', 'education', 'female_share']

fig2_labels = {
    'Sub-Saharan Africa': 'Sub-Saharan\nAfrica',
    'Northern Africa and Western Asia': 'N. Africa\n& W. Asia',
    'South Asia': 'South\nAsia',
    'Eastern and South-eastern Asia': 'E. & SE\nAsia',
    'Europe': 'Europe',
    'Latin America and the Caribbean': 'Latin America\n& Caribbean',
    'Northern America and Oceania': 'N. America\n& Oceania',
}

fig, axes = plt.subplots(2, 2, figsize=(14.4, 9.0), sharey=False, constrained_layout=True)
axes = axes.ravel()
for ax, var in zip(axes, var_order):
    d = regional_plot[regional_plot['variable'].eq(var)].sort_values('region').reset_index(drop=True)
    yvals = np.arange(len(d))
    point_colors = [region_colors[r] for r in d['region']]
    ax.errorbar(d['pct_change_per_sd'], yvals,
                xerr=[d['pct_change_per_sd'] - d['pct_ci_low'], d['pct_ci_high'] - d['pct_change_per_sd']],
                fmt='none', ecolor='#666d73', elinewidth=1.0, capsize=2, zorder=1)
    ax.scatter(d['pct_change_per_sd'], yvals, s=62, color=point_colors, edgecolor='white', linewidth=.65, zorder=2)
    ax.axvline(0, color=DARK, linewidth=.85)
    ax.set_title(var_labels[var], fontsize=12.5, fontweight='bold')
    ax.set_xlabel('% change per 1 SD')
    ax.set_yticks(yvals)
    ax.set_yticklabels([fig2_labels[r] for r in d['region']], fontsize=9.5)
    for tick, region in zip(ax.get_yticklabels(), d['region']):
        tick.set_color(DARK)
        tick.set_fontweight('normal')
    ax.invert_yaxis()
    clean_axis(ax)

    xmin = min(d['pct_ci_low'].min() - 10, -68)
    xmax = max(d['pct_ci_high'].max() + 28, 70)
    ax.set_xlim(xmin, xmax)
    x_num = xmax - 10
    x_star = xmax - 1.8
    ax.text(x_num, -0.74, 'Effect', ha='right', va='bottom', fontsize=8.7, fontweight='bold', color=DARK)
    for yidx, (_, row) in enumerate(d.iterrows()):
        ax.text(x_num, yidx, f"{row['pct_change_per_sd']:+.1f}%", ha='right', va='center', fontsize=8.3, color=DARK,
                bbox=dict(facecolor='white', edgecolor='none', alpha=.88, pad=1.0))
        if row['stars_clean']:
            ax.text(x_star, yidx, row['stars_clean'], ha='left', va='center', fontsize=12, color=RED, fontweight='bold')

fig.suptitle('Regional heterogeneity in M4 social determinants', y=1.02, fontsize=14)
fig.savefig(OUT_FIG / 'Figure_RQ3_2_regional_M4_social_coefficients.png', bbox_inches='tight')
plt.show()


In [ ]:

# Figure 3: region-specific interactions. All estimates shown; FDR-significant results highlighted.
exclude_main_interactions = ['Age x Female share', 'SES x Household size']
main_interaction_order = [it for it in interaction_order if it not in exclude_main_interactions]
int_plot = region_interactions[~region_interactions['interaction'].isin(exclude_main_interactions)].copy()
int_plot['region'] = pd.Categorical(int_plot['region'], categories=region_order, ordered=True)
int_plot['interaction'] = pd.Categorical(int_plot['interaction'], categories=main_interaction_order, ordered=True)
int_plot = int_plot.sort_values(['region','interaction']).reset_index(drop=True)
int_plot['fdr_all'] = int_plot['fdr_all_region_interactions_reject_0_05'].astype(bool)
int_plot['fdr_region'] = int_plot['fdr_within_region_reject_0_05'].astype(bool)

fig, ax = plt.subplots(figsize=(21.5, 12.6), constrained_layout=True)
regions = region_order
interactions = main_interaction_order
x_map = {r:i for i,r in enumerate(regions)}
y_map = {it:i for i,it in enumerate(interactions[::-1])}

# Encode sign by marker and significance by color/size.
for _, row in int_plot.iterrows():
    x = x_map[row['region']]
    y = y_map[row['interaction']]
    coef = row['coef_interaction_log_points']
    is_fdr = bool(row['fdr_all'])
    is_region_fdr = bool(row['fdr_region'])
    raw = row['p_value'] < 0.05
    if is_fdr:
        color = region_colors[row['region']]
        edge = DARK
        size = 470
        alpha = 1
        lw = 1.25
    elif is_region_fdr:
        color = mpl.colors.to_rgba(region_colors[row['region']], .55)
        edge = DARK
        size = 370
        alpha = .9
        lw = .9
    elif raw:
        color = '#bfc5cc'
        edge = '#798087'
        size = 260
        alpha = .85
        lw = .5
    else:
        color = '#e0e3e6'
        edge = '#c2c6ca'
        size = 155
        alpha = .65
        lw = .3
    marker = '^' if coef > 0 else 'v'
    ax.scatter(x, y, s=size, marker=marker, color=color, edgecolor=edge, linewidth=lw, alpha=alpha, zorder=3)
    if is_fdr:
        txt = f"{coef:+.2f}\n{fmt_q(row['q_value_fdr_all_region_interactions'])}"
        ax.text(x, y + .33, txt, ha='center', va='bottom', fontsize=18, fontweight='bold', color=DARK)
    elif is_region_fdr:
        txt = f"{coef:+.2f}\nwithin {fmt_q(row['q_value_fdr_within_region_8_tests'])}"
        ax.text(x, y + .30, txt, ha='center', va='bottom', fontsize=16, color=DARK)

ax.set_xticks(range(len(regions)))
ax.set_xticklabels([short_labels[r] for r in regions], rotation=0, ha='center')
ax.tick_params(axis='x', labelsize=19)
ax.set_yticks(range(len(interactions)))
interaction_label_map = {
    'Rurality x SES': 'Rurality ×\nSES',
    'Rurality x Education': 'Rurality ×\nEducation',
    'Female share x SES': 'Female share ×\nSES',
    'Female share x Education': 'Female share ×\nEducation',
    'SES x Education': 'SES ×\nEducation',
    'Female share x Rurality': 'Female share ×\nRurality',
}
ax.set_yticklabels([interaction_label_map.get(it, it.replace(' x ', ' × ')) for it in interactions[::-1]], fontsize=20)
ax.set_xlim(-.6, len(regions)-.4)
ax.set_ylim(-.72, len(interactions)-.02)
ax.set_title('Region-specific interaction effects', fontsize=28, pad=20)
ax.set_xlabel('Analytical region', fontsize=23)
ax.set_ylabel('Interaction term', fontsize=23)
ax.grid(color='#edf0f2', linewidth=1.1)
for spine in ax.spines.values():
    spine.set_visible(False)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='^', color='w', label='Positive interaction', markerfacecolor='#8baec8', markeredgecolor=DARK, markersize=20),
    Line2D([0], [0], marker='v', color='w', label='Negative interaction', markerfacecolor='#8baec8', markeredgecolor=DARK, markersize=20),
    Line2D([0], [0], marker='o', color='w', label='FDR across all region-interaction tests', markerfacecolor=BLUE, markeredgecolor=DARK, markersize=20),
    Line2D([0], [0], marker='o', color='w', label='FDR within region only', markerfacecolor=mpl.colors.to_rgba(ORANGE, .55), markeredgecolor=DARK, markersize=20),
    Line2D([0], [0], marker='o', color='w', label='Raw p<0.05 only', markerfacecolor='#bfc5cc', markeredgecolor='#798087', markersize=16),
]
ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1.0), frameon=True, title='Encoding', fontsize=19, title_fontsize=20)
fig.savefig(OUT_FIG / 'Figure_RQ3_3_region_interactions_M4_FDR.png', bbox_inches='tight')
plt.show()


In [ ]:

# Compact SI tables.
# SI 1: diagnostics summary, not a pile of separate diagnostic tables.
diag_rows = []
diag_rows.append({'Section': 'Sample', 'Metric': 'Complete RQ3 model sample', 'Value': str(int(global_m4['n_obs'].iloc[0])), 'Interpretation': 'Used for selected M4 main model.'})
diag_rows.append({'Section': 'Sample', 'Metric': 'Countries', 'Value': str(int(global_m4['n_countries'].iloc[0])), 'Interpretation': 'Country fixed effects and country-clustered SE.'})
diag_rows.append({'Section': 'VIF', 'Metric': 'Maximum M4 VIF', 'Value': f"{si_vif['VIF'].max():.2f}", 'Interpretation': 'Below conventional concern thresholds for the selected M4 specification.'})
for _, row in si_moran.iterrows():
    diag_rows.append({'Section': 'Spatial autocorrelation', 'Metric': f"Moran I, {row['weights']}", 'Value': f"I={row['moran_I']:.3f}, p={fmt_p(row['p_sim'])}", 'Interpretation': 'Residual spatial autocorrelation motivates spatial error sensitivity.'})
si_diag_summary = pd.DataFrame(diag_rows)
save_table(si_diag_summary, OUT_TABLE_SI / 'Table_SI_RQ3_1_core_diagnostics_summary.csv')

# SI 2: model stability, core variables only.
stab = si_stability[si_stability['variable'].isin(social_vars + control_vars)].copy()
stab['Variable'] = stab['variable'].map(var_labels)
stab['Effect per 1 SD'] = stab.apply(fmt_effect, axis=1)
stab['p'] = stab['p_value'].map(fmt_p)
stab = stab[['model', 'Variable', 'Effect per 1 SD', 'p', 'n_obs', 'n_countries', 'r_squared', 'adj_r_squared']]
save_table(stab, OUT_TABLE_SI / 'Table_SI_RQ3_2_global_model_stability.csv')

# SI 3: spatial sensitivity.
spat = si_spatial.copy()
spat['Variable'] = spat['variable'].map(var_labels)
spat['Effect per 1 SD'] = spat['pct_change_per_sd'].map(lambda x: f'{x:+.1f}%')
spat['p'] = spat['p_value'].map(fmt_p)
spat = spat[['model', 'Variable', 'coef_log_points_per_sd', 'std_error', 'Effect per 1 SD', 'p', 'stars', 'n_obs', 'n_countries']]
save_table(spat, OUT_TABLE_SI / 'Table_SI_RQ3_3_spatial_error_sensitivity.csv')

# SI 4: fractional logit robustness, global plus regional female/education only to keep compact.
frac_core = si_frac[(si_frac['region'].eq('Global')) | (si_frac['variable'].isin(['female_share','education']))].copy()
frac_core['Variable'] = frac_core['variable'].map(var_labels)
frac_core['AME per 1 SD'] = frac_core.apply(lambda r: f"{r['AME_per_sd']:+.3f} [{r['AME_ci_low_approx']:+.3f}, {r['AME_ci_high_approx']:+.3f}] {'' if pd.isna(r['stars']) else str(r['stars'])}".strip(), axis=1)
frac_core['p'] = frac_core['p_value'].map(fmt_p)
frac_core = frac_core[['outcome', 'region', 'Variable', 'AME per 1 SD', 'p', 'n_obs', 'n_countries']]
save_table(frac_core, OUT_TABLE_SI / 'Table_SI_RQ3_4_fractional_logit_robustness.csv')

# SI 5: global interactions among variables included in the selected M4 main model.
exclude_main_interactions = ['Age x Female share', 'SES x Household size']
gi = si_global_interactions[~si_global_interactions['interaction'].isin(exclude_main_interactions)].copy()
gi['Interaction'] = gi['interaction'].str.replace(' x ', ' × ', regex=False)
gi['Coefficient'] = gi['coef_interaction_log_points'].map(lambda x: f'{x:+.3f}')
gi['p'] = gi['p_value'].map(fmt_p)
gi['FDR q'] = gi['q_value_fdr_across_8_tests'].map(fmt_p)
gi = gi[['Interaction', 'Coefficient', 'cluster_se', 'p', 'FDR q', 'fdr_reject_0_05', 'n_obs', 'n_countries']]
save_table(gi, OUT_TABLE_SI / 'Table_SI_RQ3_5_global_interactions_M4_only.csv')

# SI 6: full regional interactions among M4 variables, because this supports Figure 3.
ri = region_interactions[~region_interactions['interaction'].isin(exclude_main_interactions)].copy()
ri['Region'] = ri['region']
ri['Interaction'] = ri['interaction'].str.replace(' x ', ' × ', regex=False)
ri['Coefficient'] = ri['coef_interaction_log_points'].map(lambda x: f'{x:+.3f}')
ri['p'] = ri['p_value'].map(fmt_p)
ri['FDR q within region'] = ri['q_value_fdr_within_region_8_tests'].map(fmt_p)
ri['FDR q all region-interactions'] = ri['q_value_fdr_all_region_interactions'].map(fmt_p)
ri = ri[['Region', 'Interaction', 'Coefficient', 'cluster_se', 'p', 'FDR q within region', 'FDR q all region-interactions', 'fdr_within_region_reject_0_05', 'fdr_all_region_interactions_reject_0_05', 'n_obs', 'n_countries']]
save_table(ri, OUT_TABLE_SI / 'Table_SI_RQ3_6_region_interactions_M4_only.csv')


In [ ]:

# Compact SI figures.
# SI Figure 1: diagnostics, correlation plus VIF.
main_vars = ['log_sufficiency_ratio','SES_raw','rural','education','female_share','log_GDP_pc','log_population','total_HDD','total_CDD']
corr = si_corr.loc[main_vars, main_vars]
pvals = si_corr_p.loc[main_vars, main_vars]
corr_labels = {'log_sufficiency_ratio':'Log ratio', **var_labels}
annot = corr.copy().astype(str)
for a in main_vars:
    for b in main_vars:
        annot.loc[a,b] = '1.00' if a == b else f"{corr.loc[a,b]:.2f}{star(pvals.loc[a,b])}"
fig, axes = plt.subplots(1, 2, figsize=(12.6, 5.4), gridspec_kw={'width_ratios':[1.45, .75]}, constrained_layout=True)
sns.heatmap(corr.rename(index=corr_labels, columns=corr_labels), annot=annot.rename(index=corr_labels, columns=corr_labels), fmt='', cmap='vlag', center=0, vmin=-1, vmax=1, square=True, linewidths=.25, cbar_kws={'label':'Pearson r', 'shrink':.75}, ax=axes[0])
axes[0].set_title('Main variable correlations')
vif_plot = si_vif.copy().sort_values('VIF')
vif_plot['label'] = vif_plot['variable'].map(var_labels)
axes[1].barh(vif_plot['label'], vif_plot['VIF'], color=BLUE)
axes[1].axvline(5, color=RED, linestyle='--', linewidth=1)
axes[1].set_xlabel('VIF')
axes[1].set_title('M4 VIF')
axes[1].grid(axis='x', color='#e6e8eb')
fig.savefig(OUT_FIG_SI / 'Figure_SI_RQ3_1_diagnostics_correlation_vif.png', bbox_inches='tight')
plt.show()

# SI Figure 2: model stability core coefficients.
stab_plot = si_stability[si_stability['variable'].isin(social_vars)].copy()
stab_plot['Variable'] = stab_plot['variable'].map(var_labels)
model_order = ['M1_full_age_hhsize','M2_drop_age','M3_drop_hhsize','M4_drop_age_hhsize']
stab_plot['model'] = pd.Categorical(stab_plot['model'], categories=model_order, ordered=True)
fig, ax = plt.subplots(figsize=(8.6, 4.8), constrained_layout=True)
offsets = np.linspace(-.24, .24, len(model_order))
y_base = {v:i for i,v in enumerate([var_labels[x] for x in social_vars][::-1])}
model_colors = dict(zip(model_order, [BLUE, ORANGE, PURPLE, TEAL]))
for mi, model in enumerate(model_order):
    d = stab_plot[stab_plot['model'].eq(model)]
    for _, row in d.iterrows():
        y = y_base[row['Variable']] + offsets[mi]
        ax.plot([row['pct_ci_low'], row['pct_ci_high']], [y,y], color=model_colors[model], lw=1.1, alpha=.85)
        ax.scatter(row['pct_change_per_sd'], y, color=model_colors[model], s=42, edgecolor='white', linewidth=.5, label=model if row['Variable'] == list(y_base.keys())[0] else None)
ax.axvline(0, color=DARK, lw=.85)
ax.set_yticks(list(y_base.values()))
ax.set_yticklabels(list(y_base.keys()))
ax.set_xlabel('Percent change in sufficiency ratio per 1 SD increase')
ax.set_title('Global social coefficient stability across factor-selection models')
ax.legend(frameon=True, loc='lower right')
clean_axis(ax)
fig.savefig(OUT_FIG_SI / 'Figure_SI_RQ3_2_model_stability.png', bbox_inches='tight')
plt.show()

# SI Figure 3: spatial sensitivity coefficient comparison.
sp = si_spatial[si_spatial['variable'].isin(social_vars + control_vars)].copy()
sp['Variable'] = sp['variable'].map(var_labels)
sp['model_short'] = np.where(sp['model'].str.contains('Spatial error'), 'Spatial error', 'Country-FE OLS')
var_order_sp = [var_labels[v] for v in (social_vars + control_vars)][::-1]
fig, ax = plt.subplots(figsize=(8.1, 5.2), constrained_layout=True)
for model, color, offset in [('Country-FE OLS', BLUE, -.13), ('Spatial error', ORANGE, .13)]:
    d = sp[sp['model_short'].eq(model)]
    for _, row in d.iterrows():
        y = var_order_sp.index(row['Variable']) + offset
        # Approx CI from std_error for both OLS/SEM comparison.
        lo = row['coef_log_points_per_sd'] - 1.96 * row['std_error']
        hi = row['coef_log_points_per_sd'] + 1.96 * row['std_error']
        ax.plot([100*(np.exp(lo)-1), 100*(np.exp(hi)-1)], [y,y], color=color, lw=1.1)
        ax.scatter(row['pct_change_per_sd'], y, color=color, s=46, edgecolor='white', linewidth=.5, label=model if row['Variable'] == var_order_sp[0] else None)
ax.axvline(0, color=DARK, lw=.85)
ax.set_yticks(range(len(var_order_sp)))
ax.set_yticklabels(var_order_sp)
ax.set_xlabel('Percent change per 1 SD increase')
ax.set_title('Spatial error sensitivity for selected M4 model')
ax.legend(frameon=True, loc='lower right')
clean_axis(ax)
fig.savefig(OUT_FIG_SI / 'Figure_SI_RQ3_3_spatial_error_sensitivity.png', bbox_inches='tight')
plt.show()

# SI Figure 4: fractional logit global AME.
fg = si_frac[(si_frac['region'].eq('Global')) & (si_frac['variable'].isin(social_vars))].copy()
fg['Variable'] = fg['variable'].map(var_labels)
fig, ax = plt.subplots(figsize=(7.6, 4.2), constrained_layout=True)
outcomes = ['sufficiency_gap','sufficiency_deficit']
outcome_colors = {'sufficiency_gap': BLUE, 'sufficiency_deficit': RED}
y_base = {v:i for i,v in enumerate([var_labels[x] for x in social_vars][::-1])}
for oi, outcome in enumerate(outcomes):
    d = fg[fg['outcome'].eq(outcome)]
    offset = -.12 if oi == 0 else .12
    for _, row in d.iterrows():
        y = y_base[row['Variable']] + offset
        ax.plot([row['AME_ci_low_approx'], row['AME_ci_high_approx']], [y,y], color=outcome_colors[outcome], lw=1.1)
        ax.scatter(row['AME_per_sd'], y, color=outcome_colors[outcome], s=44, edgecolor='white', linewidth=.5, label=outcome.replace('sufficiency_', '') if row['Variable'] == list(y_base.keys())[0] else None)
ax.axvline(0, color=DARK, lw=.85)
ax.set_yticks(list(y_base.values()))
ax.set_yticklabels(list(y_base.keys()))
ax.set_xlabel('Average marginal effect per 1 SD increase')
ax.set_title('Fractional logit robustness: gap and deficit')
ax.legend(frameon=True, loc='lower right')
clean_axis(ax)
fig.savefig(OUT_FIG_SI / 'Figure_SI_RQ3_4_fractional_logit_global_AME.png', bbox_inches='tight')
plt.show()

# SI Figure 5: global interaction terms among variables included in selected M4.
exclude_main_interactions = ['Age x Female share', 'SES x Household size']
gi_plot = si_global_interactions[~si_global_interactions['interaction'].isin(exclude_main_interactions)].copy().sort_values('coef_interaction_log_points')
fig, ax = plt.subplots(figsize=(7.4, 4.9), constrained_layout=True)
y = np.arange(len(gi_plot))
colors = np.where(gi_plot['p_value'] < .05, PURPLE, GREY)
ax.errorbar(gi_plot['coef_interaction_log_points'], y,
            xerr=[gi_plot['coef_interaction_log_points'] - gi_plot['ci_low'], gi_plot['ci_high'] - gi_plot['coef_interaction_log_points']],
            fmt='none', ecolor='#5f666d', capsize=2, linewidth=1)
ax.scatter(gi_plot['coef_interaction_log_points'], y, s=58, color=colors, edgecolor='white', linewidth=.6)
ax.axvline(0, color=DARK, lw=.85)
ax.set_yticks(y)
ax.set_yticklabels(gi_plot['interaction'].str.replace(' x ', ' × ', regex=False))
ax.set_xlabel('Interaction coefficient, log points')
ax.set_title('Global M4 interactions among selected variables')
for yy, (_, row) in enumerate(gi_plot.iterrows()):
    if row['p_value'] < .1:
        ax.text(row['ci_high'] + .012, yy, f"{row['stars_raw']} / FDR {fmt_p(row['q_value_fdr_across_8_tests'])}", va='center', fontsize=8, color=RED)
clean_axis(ax)
fig.savefig(OUT_FIG_SI / 'Figure_SI_RQ3_5_global_interactions_all_8.png', bbox_inches='tight')
plt.show()


In [ ]:

# Manifest and image sanity checks.
all_figs = sorted(list(OUT_FIG.glob('Figure_RQ3_*.png')) + list(OUT_FIG_SI.glob('Figure_SI_RQ3_*.png')))
image_checks = []
for p in all_figs:
    img = Image.open(p)
    image_checks.append({'file': str(p), 'width': img.size[0], 'height': img.size[1], 'mode': img.mode})
image_checks = pd.DataFrame(image_checks)
image_checks.to_csv(PROCESS_DIR / 'RQ3_final_figure_image_checks.csv', index=False)

manifest = {
    'notebook': str(PROJECT / 'RQ3_final_results_packaging.ipynb'),
    'main_tables_dir': str(OUT_TABLE),
    'main_figures_dir': str(OUT_FIG),
    'si_tables_dir': str(OUT_TABLE_SI),
    'si_figures_dir': str(OUT_FIG_SI),
    'main_tables': sorted([str(p) for p in OUT_TABLE.glob('Table_RQ3_*.csv')]),
    'main_figures': sorted([str(p) for p in OUT_FIG.glob('Figure_RQ3_*.png')]),
    'si_tables': sorted([str(p) for p in OUT_TABLE_SI.glob('Table_SI_RQ3_*.csv')]),
    'si_figures': sorted([str(p) for p in OUT_FIG_SI.glob('Figure_SI_RQ3_*.png')]),
    'notes': {
        'models_refit': False,
        'source_results': [str(SOURCE_MAIN), str(SOURCE_SI)],
        'interaction_visualization': 'All region-specific interactions are shown; FDR across all region-interaction tests is emphasized, within-region FDR is secondary.'
    }
}
(PROCESS_DIR / 'RQ3_final_results_packaging_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
image_checks
